# Comprehensive Data Analysis: Sensor Readings and Hospital Wait Times

Prepared in a senior data analyst style, but explained with a light, practical gist. Think of each dataset as its own little mystery: first we inspect the scene, then clean the labels, then look for patterns, oddballs, and useful business takeaways.

**Files analyzed**

- `week2_sensor_readings.csv`
- `hospital_wait_times.csv`

**Notebook flow**

1. Load libraries and settings.
2. Analyze the sensor dataset on its own.
3. Analyze the hospital dataset on its own.
4. Close with practical recommendations.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = Path(".")
SENSOR_FILE = DATA_DIR / "week2_sensor_readings.csv"
HOSPITAL_FILE = DATA_DIR / "hospital_wait_times.csv"

## Dataset 1: Week 2 Sensor Readings

### 1. Load the data

**Fun gist:** We are opening the control-room logbook. Before we trust it, we peek at the first few rows and ask: "What are we dealing with?"

In [ ]:
sensor_df = pd.read_csv(SENSOR_FILE)
sensor_df.head()

In [ ]:
sensor_df.info()

### 2. Basic profile

**Fun gist:** This is the dataset's passport photo: size, timeline, duplicate count, and the basic pulse of the readings.

| metric | value |
| --- | --- |
| rows | 2,000 |
| columns | 9 |
| start timestamp | 2026-01-06 00:00:00 |
| end timestamp | 2026-07-06 22:35:00 |
| duplicate rows | 0 |
| reading mean | 50.45 |
| reading median | 50.45 |
| reading std dev | 9.88 |
| reading min | 17.59 |
| reading max | 88.53 |
| IQR lower fence | 24.19 |
| IQR upper fence | 76.41 |
| IQR outlier count | 17 |

In [ ]:
sensor_df["timestamp"] = pd.to_datetime(sensor_df["timestamp"])

sensor_profile = pd.DataFrame({
    "metric": [
        "rows", "columns", "start timestamp", "end timestamp",
        "duplicate rows", "reading mean", "reading median",
        "reading std dev", "reading min", "reading max"
    ],
    "value": [
        sensor_df.shape[0], sensor_df.shape[1],
        sensor_df["timestamp"].min(), sensor_df["timestamp"].max(),
        sensor_df.duplicated().sum(),
        sensor_df["reading"].mean(), sensor_df["reading"].median(),
        sensor_df["reading"].std(), sensor_df["reading"].min(), sensor_df["reading"].max()
    ]
})
sensor_profile

### 3. Missing values

**Fun gist:** Missing values are like blank boxes on a checklist. Here we count the blanks before making any bold claims.

|  | missing_values |
| --- | --- |
| timestamp | 0 |
| sensor_id | 0 |
| reading | 0 |
| location | 0 |
| shift | 0 |
| hour | 0 |
| date | 0 |
| location_clean | 0 |
| sensor_clean | 0 |

In [ ]:
sensor_df.isna().sum().to_frame("missing_values")

### 4. Category consistency check

**Fun gist:** The data is speaking in nicknames: `MBA` and `mombasa` mean Mombasa, while `P1` means `Pressure_01`. We standardize names so the same thing does not wear three hats.

**Raw locations**

| raw_location | rows |
| --- | --- |
| Kisumu | 341 |
| mombasa | 339 |
| Nairobi | 339 |
| MBA | 336 |
| Mombasa | 330 |
| NRB | 315 |

**Cleaned locations**

| clean_location | rows |
| --- | --- |
| Mombasa | 1005 |
| Nairobi | 654 |
| Kisumu | 341 |

**Raw sensors**

| raw_sensor | rows |
| --- | --- |
| TEMP_SENS | 517 |
| temp_sensor | 516 |
| Pressure_01 | 484 |
| P1 | 483 |

**Cleaned sensors**

| clean_sensor | rows |
| --- | --- |
| temp_sensor | 1033 |
| Pressure_01 | 967 |

sensor_df["location_clean"] = sensor_df["location"].replace({
    "mombasa": "Mombasa",
    "MBA": "Mombasa",
    "NRB": "Nairobi"
})

sensor_df["sensor_clean"] = sensor_df["sensor_id"].replace({
    "P1": "Pressure_01",
    "TEMP_SENS": "temp_sensor"
})

display(sensor_df["location"].value_counts().to_frame("raw_count"))
display(sensor_df["location_clean"].value_counts().to_frame("clean_count"))
display(sensor_df["sensor_id"].value_counts().to_frame("raw_count"))
display(sensor_df["sensor_clean"].value_counts().to_frame("clean_count"))

### 5. Reading patterns by shift

**Fun gist:** Now we ask whether readings behave differently by work shift. This is the "does the night shift have a different soundtrack?" question.

| shift | count | mean | median | std |
| --- | --- | --- | --- | --- |
| Afternoon | 632 | 49.9 | 49.84 | 9.73 |
| Morning | 639 | 51.03 | 51.15 | 10.38 |
| Night | 729 | 50.42 | 50.13 | 9.55 |

In [ ]:
sensor_shift_summary = (
    sensor_df.groupby("shift")["reading"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
sensor_shift_summary

### 6. Reading patterns by cleaned location and sensor

**Fun gist:** We now compare the cleaned combinations. This is where messy labels stop splitting the vote.

| location_clean | sensor_clean | count | mean | median | std |
| --- | --- | --- | --- | --- | --- |
| Kisumu | Pressure_01 | 157 | 49.78 | 49.6 | 10.08 |
| Kisumu | temp_sensor | 184 | 51.58 | 51.86 | 9.34 |
| Mombasa | Pressure_01 | 500 | 50.5 | 50.89 | 9.38 |
| Mombasa | temp_sensor | 505 | 50.42 | 50.26 | 9.7 |
| Nairobi | Pressure_01 | 310 | 50.3 | 49.92 | 10.27 |
| Nairobi | temp_sensor | 344 | 50.26 | 50.01 | 10.69 |

In [1]:
sensor_combo_summary = (
    sensor_df.groupby(["location_clean", "sensor_clean"])["reading"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
sensor_combo_summary

NameError: name 'sensor_df' is not defined

### 7. Outlier check

**Fun gist:** Outliers are readings standing dramatically in the doorway. Some are real incidents, some are sensor hiccups. We flag them instead of deleting them blindly.

| timestamp | sensor_id | reading | location | shift |
| --- | --- | --- | --- | --- |
| 2026-01-06 06:10:00 | Pressure_01 | 23.8 | Nairobi | Night |
| 2026-01-06 14:55:00 | TEMP_SENS | 77.2 | MBA | Morning |
| 2026-01-06 17:25:00 | TEMP_SENS | 88.53 | Nairobi | Morning |
| 2026-01-06 21:50:00 | temp_sensor | 17.59 | Mombasa | Afternoon |
| 2026-02-06 15:50:00 | TEMP_SENS | 80.79 | NRB | Morning |
| 2026-03-06 05:50:00 | TEMP_SENS | 23.03 | NRB | Afternoon |
| 2026-03-06 07:40:00 | TEMP_SENS | 23.49 | NRB | Morning |
| 2026-04-06 16:25:00 | temp_sensor | 21.51 | Nairobi | Afternoon |

In [2]:
q1 = sensor_df["reading"].quantile(0.25)
q3 = sensor_df["reading"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

sensor_outliers = sensor_df[
    (sensor_df["reading"] < lower_fence) |
    (sensor_df["reading"] > upper_fence)
]

print(f"Lower fence: {lower_fence:.2f}")
print(f"Upper fence: {upper_fence:.2f}")
print(f"Outlier rows: {len(sensor_outliers)}")
sensor_outliers.head(8)

NameError: name 'sensor_df' is not defined

### 8. Visual exploration

**Fun gist:** Tables tell us the facts; charts help the facts wave at us from across the room.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(sensor_df["reading"], kde=True, ax=axes[0, 0], color="#3a86ff")
axes[0, 0].set_title("Distribution of Sensor Readings")

sns.boxplot(data=sensor_df, x="sensor_clean", y="reading", ax=axes[0, 1])
axes[0, 1].set_title("Readings by Cleaned Sensor")
axes[0, 1].tick_params(axis="x", rotation=20)

sns.countplot(data=sensor_df, x="location_clean", ax=axes[1, 0])
axes[1, 0].set_title("Record Count by Cleaned Location")

sensor_hourly = sensor_df.assign(hour=sensor_df["timestamp"].dt.hour).groupby("hour")["reading"].mean()
sensor_hourly.plot(marker="o", ax=axes[1, 1], color="#ff6b35")
axes[1, 1].set_title("Average Reading by Hour of Day")
axes[1, 1].set_ylabel("Average reading")

plt.tight_layout()
plt.show()

### Sensor dataset takeaways

- The file has a complete, regular 5-minute time series with no missing values.
- The biggest data quality issue is inconsistent naming for locations and sensors.
- The readings are centered close to 50, matching the synthetic generation logic.
- Outliers exist, but they should be reviewed rather than automatically removed.
- Cleaned categories make summaries much more reliable because `MBA`, `mombasa`, and `Mombasa` stop being treated as separate places.

---

## Dataset 2: Hospital Wait Times

### 1. Load the data

**Fun gist:** This is the hospital queue board. We check who arrived, where they went, how urgent they were, how long they waited, and who was on duty.

In [ ]:
hospital_df = pd.read_csv(HOSPITAL_FILE)
hospital_df.head()

In [ ]:
hospital_df.info()

### 2. Basic profile

**Fun gist:** First we take the hospital dataset's vital signs: rows, time span, patient count, and wait-time behavior.

| metric | value |
| --- | --- |
| rows | 3,000 |
| columns | 10 |
| start timestamp | 2026-01-01 00:10:00 |
| end timestamp | 2026-01-30 23:17:00 |
| duplicate rows | 0 |
| unique patients | 2,564 |
| wait mean | 45.61 |
| wait median | 45.44 |
| wait std dev | 15.51 |
| wait min | -13.80 |
| wait max | 108.91 |
| negative waits | 5 |
| IQR lower fence | 3.88 |
| IQR upper fence | 87.58 |
| IQR outlier count | 22 |

In [ ]:
hospital_df["timestamp"] = pd.to_datetime(hospital_df["timestamp"])

hospital_profile = pd.DataFrame({
    "metric": [
        "rows", "columns", "start timestamp", "end timestamp",
        "duplicate rows", "unique patients", "wait mean", "wait median",
        "wait std dev", "wait min", "wait max"
    ],
    "value": [
        hospital_df.shape[0], hospital_df.shape[1],
        hospital_df["timestamp"].min(), hospital_df["timestamp"].max(),
        hospital_df.duplicated().sum(), hospital_df["patient_id"].nunique(),
        hospital_df["wait_time_min"].mean(), hospital_df["wait_time_min"].median(),
        hospital_df["wait_time_min"].std(), hospital_df["wait_time_min"].min(),
        hospital_df["wait_time_min"].max()
    ]
})
hospital_profile

### 3. Missing values

**Fun gist:** Before drawing conclusions, we check whether any patient records have blank pockets.

|  | missing_values |
| --- | --- |
| timestamp | 0 |
| patient_id | 0 |
| ward_location | 0 |
| priority | 0 |
| wait_time_min | 0 |
| nurse_on_duty | 0 |
| date | 0 |
| hour | 0 |
| ward_clean | 0 |
| priority_clean | 0 |

In [ ]:
hospital_df.isna().sum().to_frame("missing_values")

### 4. Category consistency check

**Fun gist:** The hospital data uses aliases too: `ER` and `emergency` are Emergency, `Peds` is Pediatrics, and `GW` is General Ward. Priority also has urgent shouting in two different ways.

**Raw wards**

| raw_ward | rows |
| --- | --- |
| ER | 443 |
| Emergency | 436 |
| GW | 433 |
| General Ward | 431 |
| Pediatrics | 431 |
| Peds | 420 |
| emergency | 406 |

**Cleaned wards**

| clean_ward | rows |
| --- | --- |
| Emergency | 1285 |
| General Ward | 864 |
| Pediatrics | 851 |

**Raw priorities**

| raw_priority | rows |
| --- | --- |
| Medium | 613 |
| URGENT | 610 |
| urgent | 609 |
| High | 597 |
| Low | 571 |

**Cleaned priorities**

| clean_priority | rows |
| --- | --- |
| Urgent | 1219 |
| Medium | 613 |
| High | 597 |
| Low | 571 |

In [ ]:
hospital_df["ward_clean"] = hospital_df["ward_location"].replace({
    "ER": "Emergency",
    "emergency": "Emergency",
    "Peds": "Pediatrics",
    "GW": "General Ward"
})

hospital_df["priority_clean"] = hospital_df["priority"].replace({
    "urgent": "Urgent",
    "URGENT": "Urgent"
})

display(hospital_df["ward_location"].value_counts().to_frame("raw_count"))
display(hospital_df["ward_clean"].value_counts().to_frame("clean_count"))
display(hospital_df["priority"].value_counts().to_frame("raw_count"))
display(hospital_df["priority_clean"].value_counts().to_frame("clean_count"))

### 5. Wait time by ward

**Fun gist:** This tells us where the waiting-room traffic feels heaviest. Count shows workload; mean and median show wait experience.

| ward_clean | count | mean | median | std |
| --- | --- | --- | --- | --- |
| Emergency | 1285 | 45.65 | 45.42 | 15.4 |
| General Ward | 864 | 44.81 | 44.54 | 15.37 |
| Pediatrics | 851 | 46.37 | 46.59 | 15.79 |

In [ ]:
ward_wait_summary = (
    hospital_df.groupby("ward_clean")["wait_time_min"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
ward_wait_summary

### 6. Wait time by priority

**Fun gist:** Priority should ideally shape waiting time. Here we check whether urgent cases are actually moving faster or just wearing a louder label.

| priority_clean | count | mean | median | std |
| --- | --- | --- | --- | --- |
| High | 597 | 45.49 | 46.08 | 15.28 |
| Low | 571 | 46.34 | 46.55 | 14.84 |
| Medium | 613 | 44.93 | 44.51 | 16.19 |
| Urgent | 1219 | 45.67 | 44.97 | 15.58 |

In [ ]:
priority_wait_summary = (
    hospital_df.groupby("priority_clean")["wait_time_min"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
priority_wait_summary

### 7. Wait time by nurse on duty

**Fun gist:** This is not a blame chart. It is a staffing clue. Differences can come from shift timing, ward mix, patient load, or triage process.

| nurse_on_duty | count | mean | median | std |
| --- | --- | --- | --- | --- |
| Nurse_Joy | 768 | 45.9 | 45.49 | 15.27 |
| Nurse_Ratched | 781 | 45.27 | 44.84 | 15.49 |
| Staff_A | 713 | 46.3 | 47.14 | 15.71 |
| Staff_B | 738 | 45.01 | 45.41 | 15.59 |

In [ ]:
nurse_wait_summary = (
    hospital_df.groupby("nurse_on_duty")["wait_time_min"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)
nurse_wait_summary

### 8. Negative and unusual wait times

**Fun gist:** A negative wait time means the clock has started telling jokes. In real operations, these rows should be corrected or excluded from service metrics after investigation.

**Negative wait examples**

| timestamp | patient_id | ward_location | priority | wait_time_min | nurse_on_duty |
| --- | --- | --- | --- | --- | --- |
| 2026-01-04 06:52:00 | P-9163 | GW | High | -1.46 | Staff_B |
| 2026-01-20 17:49:00 | P-9585 | GW | urgent | -1.92 | Staff_B |
| 2026-01-05 21:27:00 | P-9754 | Emergency | Medium | -13.8 | Staff_A |
| 2026-01-19 17:25:00 | P-4916 | Pediatrics | URGENT | -3.06 | Nurse_Joy |
| 2026-01-27 12:36:00 | P-1038 | Pediatrics | Low | -0.93 | Staff_B |

**IQR outlier examples**

| timestamp | patient_id | ward_location | priority | wait_time_min | nurse_on_duty |
| --- | --- | --- | --- | --- | --- |
| 2026-01-25 01:57:00 | P-4630 | Peds | Medium | 88.32 | Staff_A |
| 2026-01-20 05:13:00 | P-3778 | Emergency | High | 3.24 | Staff_A |
| 2026-01-11 16:56:00 | P-1457 | General Ward | High | 3.64 | Nurse_Joy |
| 2026-01-03 13:03:00 | P-5434 | General Ward | High | 87.97 | Nurse_Joy |
| 2026-01-02 17:25:00 | P-9588 | Emergency | Low | 90.56 | Nurse_Joy |
| 2026-01-04 06:52:00 | P-9163 | GW | High | -1.46 | Staff_B |
| 2026-01-26 10:37:00 | P-7220 | emergency | Low | 94.07 | Nurse_Joy |
| 2026-01-14 12:18:00 | P-3503 | GW | URGENT | 88.64 | Staff_A |

In [ ]:
q1 = hospital_df["wait_time_min"].quantile(0.25)
q3 = hospital_df["wait_time_min"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

negative_waits = hospital_df[hospital_df["wait_time_min"] < 0]
hospital_outliers = hospital_df[
    (hospital_df["wait_time_min"] < lower_fence) |
    (hospital_df["wait_time_min"] > upper_fence)
]

print(f"Negative wait rows: {len(negative_waits)}")
print(f"Lower fence: {lower_fence:.2f}")
print(f"Upper fence: {upper_fence:.2f}")
print(f"Outlier rows: {len(hospital_outliers)}")

display(negative_waits.head(8))
display(hospital_outliers.head(8))

### 9. Visual exploration

**Fun gist:** Time to turn the waiting-room ledger into pictures: distribution, ward pressure, priority comparison, and nurse workload.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(hospital_df["wait_time_min"], kde=True, ax=axes[0, 0], color="#2a9d8f")
axes[0, 0].set_title("Distribution of Wait Times")

sns.boxplot(data=hospital_df, x="ward_clean", y="wait_time_min", ax=axes[0, 1])
axes[0, 1].set_title("Wait Time by Cleaned Ward")
axes[0, 1].tick_params(axis="x", rotation=20)

sns.boxplot(data=hospital_df, x="priority_clean", y="wait_time_min", ax=axes[1, 0])
axes[1, 0].set_title("Wait Time by Cleaned Priority")

sns.countplot(data=hospital_df, x="nurse_on_duty", ax=axes[1, 1])
axes[1, 1].set_title("Patient Records by Nurse on Duty")
axes[1, 1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
hospital_df["date"] = hospital_df["timestamp"].dt.date
hospital_df["hour"] = hospital_df["timestamp"].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

daily_volume = hospital_df.groupby("date").size()
daily_volume.plot(marker="o", ax=axes[0], color="#7b2cbf")
axes[0].set_title("Daily Patient Record Volume")
axes[0].set_ylabel("Patients")

hourly_wait = hospital_df.groupby("hour")["wait_time_min"].mean()
hourly_wait.plot(marker="o", ax=axes[1], color="#f77f00")
axes[1].set_title("Average Wait Time by Hour")
axes[1].set_ylabel("Average wait time, minutes")

plt.tight_layout()
plt.show()

### Hospital dataset takeaways

- The file has no missing values, but category standardization is essential.
- Emergency, ER, and emergency should be analyzed as one ward.
- Urgent and URGENT should be normalized before any priority reporting.
- Negative wait times are operationally impossible and should be investigated.
- Nurse-level comparisons are useful, but they need context before being interpreted as performance differences.
- The data is synthetic, so patterns may be evenly distributed; still, the workflow mirrors real hospital analytics.

---

## Practical Recommendations

### For the sensor dataset

1. Create official lookup tables for `sensor_id` and `location`.
2. Keep raw labels, but add cleaned reporting columns.
3. Flag outliers for review instead of deleting them immediately.
4. Monitor readings by sensor, shift, and location after standardization.

### For the hospital dataset

1. Standardize ward and priority names at data entry.
2. Treat negative wait times as data quality exceptions.
3. Track wait time by cleaned ward, priority, hour, and nurse assignment.
4. Use nurse comparisons as a starting point for workflow review, not as a standalone performance score.

**Final analyst gist:** Clean labels first, summarize second, visualize third, and only then make decisions. Messy categories can make a calm dataset look chaotic.